In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib widget

In [ ]:
import torch
from scripts.arraySpec import ArraySpec
from scripts.batchFactory import generateBatch
from ui.export_batch import export_batch_for_ui
from ui.export_target import load_target_from_pt

# Setup device and precision
dtype = torch.float32
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("cpu")
else:
    device = torch.device("cpu")


# spec = ArraySpec(failRate=0.05, positionJitterSTD=0.01, phaseJitterSTD=0.1)
spec = ArraySpec(geometry="UCA", allowedAspectRatio=[1], allowedElementCount=[40000])
batch = generateBatch(spec, batchSize=2, device=device, dtype=dtype, weightsType="random")
target = load_target_from_pt("target_spec.pt")
hotspot = target.hotspotCoordinates[0].to(device)

batch2 = generateBatch(
    spec, batchSize=2, device=device, dtype=dtype, weightsType="directed", targetLLA=hotspot
)

export_batch_for_ui(batch, "batch.json", sample_id=0)
export_batch_for_ui(batch2, "batch2.json", sample_id=0)


# Output test values
print("--- Antenna Batch Generated Successfully ---")
print(f"Elements in Array: {batch.N}")
print(f"Device           : {batch.device}")
print(f"Dtype            : {batch.dtype}")
print(f"Wavelength       : {batch.wavelength:.5f}m")
print(f"Local Positions  : {batch.elementLocalPosition.shape} | {batch.elementLocalPosition.dtype}")
print(f"Base Weights     : {batch.weights.shape} | {batch.weights.dtype}")
print(f"Effective Weights: {batch.effective_weights().shape} | {batch.effective_weights().dtype}")
print(f"Gain             : {batch.gain.shape} | {batch.gain.dtype}")
print(f"LLA [Lat,Lon,Alt]: {batch.LLAPosition.shape} | {batch.LLAPosition.dtype}")
print(f"ECEF      [X,Y,Z]: {batch.ECEFPosition.shape} | {batch.ECEFPosition.dtype}")

In [ ]:
from scripts.plots import plotArrayGeometry

plotArrayGeometry(batch2)

In [ ]:
from scripts.plots import plotArrayFactor

plotArrayFactor(batch2, projection=None, plot3d=False, xProjectionScale=20)

In [ ]:
from models.reward import batchScore

score = batchScore(batch, target)
print(score)
score = batchScore(batch2, target)
print(score)